# Session 04: 예측 API 구현

## 학습 목표

이번 세션에서는 학습된 ML 모델을 FastAPI 엔드포인트로 감싸서 **실제로 호출 가능한 `/predict` API**를 완성합니다.

### 완성할 것들
- `schemas.py`: Pydantic 기반 요청/응답 스키마 (입력 검증 자동화)
- `model.py`: 모델 로딩 + 추론 로직
- `main.py`: FastAPI 앱에 `/health`, `/predict` 엔드포인트 통합

### WHY: 왜 스키마 → 모델 → API 순서로 만드는가?
1. **스키마 먼저**: 입출력 형태를 확정해야 모델 래핑과 API 설계가 흔들리지 않는다
2. **모델 래핑**: 스키마에 맞춰 모델이 입력을 받고 결과를 반환하도록 한다
3. **API 통합**: 스키마와 모델을 FastAPI에 연결하면 자동으로 문서화·검증이 완성된다

In [ ]:
import json
import os
from pathlib import Path

## 1. 요청 스키마 설계

### WHY Pydantic `Field`를 활용하는가?

| 기능 | 효과 |
|------|------|
| `ge`, `le` 제약 | 모델 학습 범위 밖의 입력을 사전 차단 (외삽 방지) |
| `description` | Swagger 문서에 자동 반영 → 프론트엔드 개발자 소통 비용 절감 |
| `examples` | "Try it out" 시 예시값 자동 채움 |
| 타입 힌트 | 잘못된 타입 → 422 에러 자동 반환 (별도 코드 불필요) |

### WHY 영어 필드명 + 한글 description?
- API 필드명은 영어가 국제 표준이며, 다양한 클라이언트와의 호환성이 좋다
- 한글 description으로 데이터 분석가/기획자와의 커뮤니케이션 비용을 줄인다
- 모델 내부에서 영어 API 필드명을 한글 CSV 컬럼명으로 매핑하여 처리한다

In [ ]:
# 여기에 코드를 작성하세요


## 2. 응답 스키마 설명

| 필드 | 타입 | 용도 |
|------|------|------|
| `approved` | `bool` | 승인/거절 이진 판정 → 비즈니스 로직에서 바로 사용 |
| `probability` | `float` | 0.0~1.0 사이의 승인 확률 → 임계값 조정에 활용 |
| `risk_grade` | `str` | A/B/C/D 등급 → 대시보드, 리포트에 표시 |

### 리스크 등급 기준
- **A**: 확률 >= 0.75 (매우 안전)
- **B**: 확률 >= 0.50 (양호)
- **C**: 확률 >= 0.25 (주의)
- **D**: 확률 < 0.25 (위험)

## 3. 모델 로딩 클래스

### WHY 단순 클래스 + app.state 패턴인가?
- app.state에 인스턴스를 저장하면 싱글턴 패턴 없이도 메모리에 모델을 한 번만 로드할 수 있다
- 코드가 단순해지고 테스트에서 mock 주입이 용이하다

### FIELD_TO_COLUMN 매핑
- API는 영어 필드명(age, gender 등)을 사용하지만, 학습 데이터(CSV)와 인코더는 한글 컬럼명을 사용한다
- 이 매핑으로 두 세계를 연결한다

### Pipeline 구조
```
models/
├── loan_pipeline.pkl     # 학습된 sklearn Pipeline (전처리 + 모델)
├── label_encoders.pkl    # 범주형 변수 인코더 딕셔너리
└── feature_names.pkl     # 피처 이름 목록
```

### 추론 흐름
1. `LoanRequest` → dict로 변환
2. 영어 필드명 → 한글 컬럼명 매핑 (FIELD_TO_COLUMN)
3. 범주형 필드(성별, 주거형태 등)를 LabelEncoder로 숫자 변환
4. Pipeline에 입력 → `predict_proba()` 호출
5. 확률값으로 승인 여부 + 리스크 등급 결정 → `LoanResponse` 반환

In [ ]:
# 여기에 코드를 작성하세요


## 4. main.py 통합

### FastAPI Lifespan 패턴

```python
@asynccontextmanager
async def lifespan(app):
    # 서버 시작 시 실행 (모델 로딩)
    model = LoanModel()
    model.load()
    app.state.model = model
    yield
    # 서버 종료 시 실행 (리소스 정리)
```

WHY lifespan을 사용하는가?
- `@app.on_event("startup")`은 FastAPI에서 **deprecated** (더 이상 권장하지 않음)
- lifespan은 시작과 종료를 하나의 함수에서 관리 → 코드가 더 깔끔
- 비동기 컨텍스트 매니저이므로 리소스 정리가 보장됨

WHY app.state에 모델을 저장하는가?
- 싱글턴 패턴 없이도 메모리에 모델을 한 번만 로드할 수 있다
- 테스트에서 app.state.model을 교체하면 쉽게 mock 가능

In [ ]:
# 여기에 코드를 작성하세요


## 5. 테스트 방법

### 서버 실행
터미널에서:
```bash
cd service-a-loan
uvicorn app.main:app --reload --port 8000
```

### cURL로 테스트
```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "age": 35,
    "gender": "남",
    "annual_income": 5000.0,
    "employment_years": 5,
    "housing_type": "자가",
    "credit_score": 720,
    "existing_loan_count": 2,
    "annual_card_usage": 2400.0,
    "debt_ratio": 35.5,
    "loan_amount": 3000.0,
    "loan_purpose": "주택구입",
    "repayment_method": "원리금균등",
    "loan_period": 36
  }'
```

### Swagger UI
브라우저에서 http://localhost:8000/docs 접속 → "Try it out" 클릭

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


## 정리

### 이번 세션에서 만든 것

| 파일 | 역할 |
|------|------|
| `schemas.py` | 요청/응답 데이터 검증 + Swagger 자동 문서화 |
| `model.py` | 모델 로딩 + 추론 로직 (app.state 패턴) |
| `main.py` | FastAPI 앱 + lifespan + 엔드포인트 통합 |

### 핵심 개념
1. **Pydantic Field**: 제약 조건으로 입력 검증 자동화
2. **영어 필드명 + FIELD_TO_COLUMN 매핑**: API는 영어, 내부는 한글 CSV 컬럼명
3. **app.state 모델 저장**: 서버 시작 시 한 번만 로드, 모든 요청에서 재사용
4. **Lifespan**: 서버 생명주기에 맞춘 리소스 관리
5. **logging**: print() 대신 로그 레벨 구분 가능한 logging 사용
6. **자동 422 에러**: 잘못된 입력 → Pydantic이 자동으로 상세 에러 반환
